In [2]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import classification_report

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --- This script assumes 'create_window_util.py' is available ---
# It must contain your 'create_windows' function.
try:
    from create_window_util import create_windows
except ImportError:
    print("="*50)
    print("ERROR: Could not import 'create_windows'.")
    print("Please make sure 'create_window_util.py' is in the same directory.")
    print("="*50)
    exit()

# --- 1. Define Constants ---
RAW_DATA_FILE = "../processed_data/SDHAR/final_processed_data_ALL_DAYS.csv"
MODEL_SAVE_PATH = "../models/"
TARGET_COLUMN = 'activity_user_1'

# Windowing parameters
WINDOW_SIZE = 60
STEP_SIZE = 30

# Training parameters
TEST_SIZE = 0.2
RANDOM_STATE = 42
EPOCHS = 100
BATCH_SIZE = 256
LEARNING_RATE = 0.0001
VALIDATION_SPLIT = 0.1

# Class names for the final report (MUST match the integer encoding)
CLASS_NAMES = [
    "BATHROOM ACTIVITY", "CHORES", "COOK", "DISHWASHING", "DRESS", "EAT",
    "LAUNDRY", "MAKE SIMPLE FOOD", "OUT HOME", "PET", "READ", "RELAX",
    "SHOWER", "SLEEP", "TAKE MEDS", "WATCH TV", "WORK", "OTHER"
]

# --- 2. Data Loading and Processing Function ---
def load_and_process_data():
    """
    Loads the raw CSV, windows it, and splits into train/test sets.
    """
    print(f"Loading raw data from {RAW_DATA_FILE}...")
    df = pd.read_csv(RAW_DATA_FILE)

    print("Separating Features and Target...")
    df.dropna(inplace=True)
    X = df.drop(columns=[col for col in df.columns if 'activity' in col])
    y = df[TARGET_COLUMN].astype(int)  # Use 1D integer labels

    print(f"Creating sliding windows (size={WINDOW_SIZE}, step={STEP_SIZE})...")
    X_win, y_win = create_windows(X, y, WINDOW_SIZE, STEP_SIZE)
    print(f"  - Windowed X shape: {X_win.shape}")
    print(f"  - Windowed y shape: {y_win.shape}")

    print("Splitting data into training and test sets...")
    X_train_3d, X_test_3d, y_train_1d, y_test_1d = train_test_split(
        X_win, y_win,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_win
    )

    print(f"  - Original training distribution: {Counter(y_train_1d)}")
    print(f"  - Original test distribution: {Counter(y_test_1d)}")

    return X_train_3d, X_test_3d, y_train_1d, y_test_1d

# --- 3. Main Execution ---
if __name__ == "__main__":
    # Ensure model save directory exists
    if not os.path.exists(MODEL_SAVE_PATH):
        os.makedirs(MODEL_SAVE_PATH)
        print(f"Created directory: {MODEL_SAVE_PATH}")

    # --- Load and Prep Data ---
    X_train, X_test, y_train_1d, y_test_1d = load_and_process_data()

    n_timesteps = X_train.shape[1]
    n_features = X_train.shape[2]

    # Get num_classes from *all* potential labels
    all_labels = np.concatenate((y_train_1d, y_test_1d))
    num_classes = len(np.unique(all_labels))

    if len(CLASS_NAMES) != num_classes:
        print(f"Warning: CLASS_NAMES list has {len(CLASS_NAMES)} items, but data has {num_classes} classes.")

    print(f"\nData has {n_timesteps} timesteps, {n_features} features, and {num_classes} classes.")

    # One-hot encode labels for Keras
    print("One-hot encoding labels...")
    y_train = to_categorical(y_train_1d, num_classes=num_classes)
    y_test = to_categorical(y_test_1d, num_classes=num_classes)

    # --- Calculate Class Weights ---
    print("Calculating class weights...")
    class_weights_array = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_1d),
        y=y_train_1d
    )
    class_weights_dict = dict(enumerate(class_weights_array))
    print("  - Weights calculated.")

    # --- Build the Model ---
    print("Building the Bidirectional LSTM model...")
    model = Sequential([
        Bidirectional(
            LSTM(128, input_shape=(n_timesteps, n_features), return_sequences=True)
        ),
        Dropout(0.4),
        Bidirectional(LSTM(128)),
        Dropout(0.4),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

    optimizer = Adam(learning_rate=LEARNING_RATE)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()

    # --- Define Callbacks ---
    best_model_path = os.path.join(MODEL_SAVE_PATH, 'LSTM_ClassWeight_best.keras')

    early_stopper = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    model_checkpoint = ModelCheckpoint(
        filepath=best_model_path,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )

    # --- Train the Model ---
    print("\n--- Starting Model Training ---")
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        verbose=1,
        class_weight=class_weights_dict,  # Apply balancing weights
        callbacks=[early_stopper, model_checkpoint]
    )
    print(f"\n--- Training Complete ---")
    print(f"Best model saved to {best_model_path}")

    # --- Evaluate the Model ---
    print("\n--- Evaluating Model on Test Set ---")
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest Accuracy: {accuracy * 100:.2f}%")
    print(f"Test Loss: {loss:.4f}")

    y_pred_probs = model.predict(X_test)
    y_pred_labels = np.argmax(y_pred_probs, axis=1) # Convert one-hot back to 1D

    print("\nClassification Report (LSTM with Class Weights):")
    print(classification_report(y_test_1d, y_pred_labels, target_names=CLASS_NAMES, digits=4))

Loading raw data from ../processed_data/SDHAR/final_processed_data_ALL_DAYS.csv...
Separating Features and Target...
Creating sliding windows (size=60, step=30)...
  - Windowed X shape: (67467, 60, 41)
  - Windowed y shape: (67467,)
Splitting data into training and test sets...
  - Original training distribution: Counter({np.int64(8): 21012, np.int64(13): 17077, np.int64(15): 4377, np.int64(17): 2876, np.int64(5): 2561, np.int64(0): 1599, np.int64(16): 1330, np.int64(10): 703, np.int64(11): 608, np.int64(2): 537, np.int64(7): 354, np.int64(12): 340, np.int64(1): 297, np.int64(4): 127, np.int64(3): 73, np.int64(9): 64, np.int64(14): 28, np.int64(6): 10})
  - Original test distribution: Counter({np.int64(8): 5253, np.int64(13): 4270, np.int64(15): 1094, np.int64(17): 719, np.int64(5): 641, np.int64(0): 400, np.int64(16): 332, np.int64(10): 176, np.int64(11): 152, np.int64(2): 134, np.int64(7): 89, np.int64(12): 85, np.int64(1): 74, np.int64(4): 32, np.int64(3): 18, np.int64(9): 16, np.in

C:\Users\jesse\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


--- Starting Model Training ---
Epoch 1/100
190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 781ms/step - accuracy: 0.0885 - loss: 2.8360
Epoch 1: val_loss improved from None to 2.36905, saving model to ../models/LSTM_ClassWeight_best.keras
190/190 ━━━━━━━━━━━━━━━━━━━━ 161s 808ms/step - accuracy: 0.1311 - loss: 2.8155 - val_accuracy: 0.3040 - val_loss: 2.3691
Epoch 2/100
190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 575ms/step - accuracy: 0.3094 - loss: 2.5586
Epoch 2: val_loss improved from 2.36905 to 1.47593, saving model to ../models/LSTM_ClassWeight_best.keras
190/190 ━━━━━━━━━━━━━━━━━━━━ 115s 602ms/step - accuracy: 0.3925 - loss: 2.3954 - val_accuracy: 0.6062 - val_loss: 1.4759
Epoch 3/100
190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 683ms/step - accuracy: 0.5636 - loss: 2.1003
Epoch 3: val_loss improved from 1.47593 to 0.96951, saving model to ../models/LSTM_ClassWeight_best.keras
190/190 ━━━━━━━━━━━━━━━━━━━━ 136s 716ms/step - accuracy: 0.5836 - loss: 1.9769 - val_accuracy: 0.7088 - val_loss: 0.9695
Epoch 4/100
190/190 ━━

KeyboardInterrupt: 

bidirectional could improve, but need more compute. Got to roughly 85% in roughly 20 epochs
